# CNN-GNN-HMER Baseline - Kaggle 2xT4 + W&B

Notebook này được dựng lại dựa trên `template.ipynb` gốc:

- Giữ flow Miniconda → env `tamer` Python 3.7 → clone repo → `%cd` vào đúng project con → install → unzip CROHME → train.
- Không dùng `kaggle_full_*.yaml`.
- Ghi đè trực tiếp `config/crohme.yaml` bằng config chuẩn của từng mô hình.
- Train mặc định bằng 2 GPU T4: `--trainer.gpus=2`.
- W&B tách riêng project: `CNN-GNN-HMER-baseline`.
- Sau train có cell eval, upload artifact checkpoint/log/eval, và nén output ra `/kaggle/working/baseline_results.tar.gz`.

## 0. Tham số chính

Nếu Kaggle không bật 2 GPU, sửa `GPUS = 1`. Còn mặc định cho 2xT4 là `GPUS = 2`.

In [ ]:
# Tham số chính cho notebook
GPUS = 2
CONFIG = "config/crohme.yaml"
WANDB_PROJECT = "CNN-GNN-HMER-baseline"
WANDB_RUN_NAME = "baseline-crohme-2gpu-t4"
WORKDIR = "/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/0-cnn-transformer-baseline"
RESULT_DIR = "/kaggle/output/baseline_results"
RESULT_TAR = "baseline_results.tar.gz"

print("WORKDIR:", WORKDIR)
print("CONFIG:", CONFIG)
print("GPUS:", GPUS)
print("WANDB_PROJECT:", WANDB_PROJECT)
print("WANDB_RUN_NAME:", WANDB_RUN_NAME)

## 1. Cài đặt Miniconda

In [ ]:
# Cài đặt Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /kaggle/working/miniconda
!rm Miniconda3-latest-Linux-x86_64.sh

# Thêm conda vào PATH
import os
os.environ['PATH'] = "/kaggle/working/miniconda/bin:" + os.environ['PATH']

## 2. Tạo môi trường Python 3.7

In [ ]:
# Tạo môi trường Python 3.7
!/kaggle/working/miniconda/bin/conda create -n tamer python=3.7 -y

## 3. Kiểm tra phiên bản Python và pip

In [ ]:
# Kiểm tra phiên bản Python và pip
shell_script = """
source /kaggle/working/miniconda/bin/activate tamer
python --version
pip --version
"""
with open("activate_env.sh", "w") as f:
    f.write(shell_script)
!bash activate_env.sh

## 4. Clone repo CNN-GNN-HMER

In [ ]:
# Clone repo
# Nếu notebook bị restart và repo đã tồn tại, cell này sẽ bỏ qua clone để không lỗi.
!if [ ! -d "/kaggle/working/CNN-GNN-HMER" ]; then git clone https://github.com/KhaiHASO/CNN-GNN-HMER.git /kaggle/working/CNN-GNN-HMER; else echo "Repo đã tồn tại: /kaggle/working/CNN-GNN-HMER"; fi

## 5. Di chuyển vào thư mục Baseline

In [ ]:
# Di chuyển vào thư mục dự án con
%cd /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/0-cnn-transformer-baseline
!pwd
!ls -lh

## 6. Cài đặt các gói từ conda

In [ ]:
# Cài đặt các gói từ conda
!source /kaggle/working/miniconda/bin/activate tamer && conda install pytorch-lightning=1.4.9 torchmetrics=0.6.0 -c conda-forge -y
!source /kaggle/working/miniconda/bin/activate tamer && conda install pandoc=1.19.2.1 -c conda-forge -y

## 7. Fix lỗi GLIBCXX

In [ ]:
# Cài đặt libstdcxx-ng để fix lỗi GLIBCXX
!source /kaggle/working/miniconda/bin/activate tamer && conda install -c conda-forge libstdcxx-ng -y

## 8. Cài requirements, setup.py và W&B

In [ ]:
# Cài đặt các gói từ requirements.txt và setup.py
!source /kaggle/working/miniconda/bin/activate tamer && pip install -r requirements.txt && pip install -e .

# Đảm bảo có wandb để WandbLogger + artifact hoạt động
!source /kaggle/working/miniconda/bin/activate tamer && pip install wandb

## 9. Ghi đè config/crohme.yaml chuẩn cho notebook này

In [ ]:
# Backup config cũ trước khi ghi đè
!mkdir -p config_backup
!cp config/crohme.yaml config_backup/crohme.original.yaml || true

In [ ]:
%%writefile config/crohme.yaml
seed_everything: 7
trainer:
  checkpoint_callback: true
  logger:
    class_path: pytorch_lightning.loggers.WandbLogger
    init_args:
      project: CNN-GNN-HMER-baseline
      name: baseline-crohme-2gpu-t4
      save_dir: lightning_logs
  callbacks:
    - class_path: pytorch_lightning.callbacks.LearningRateMonitor
      init_args:
        logging_interval: epoch
    - class_path: pytorch_lightning.callbacks.ModelCheckpoint
      init_args:
        save_top_k: 1
        monitor: val_ExpRate
        mode: max
        filename: '{epoch}-{step}-{val_ExpRate:.4f}'
  gpus: 2
  accelerator: auto
  check_val_every_n_epoch: 2
  max_epochs: 100
  deterministic: true
  precision: 16
model:
  d_model: 256
  # encoder
  growth_rate: 24
  num_layers: 16
  # decoder
  nhead: 8
  num_decoder_layers: 3
  dim_feedforward: 1024
  dc: 32
  dropout: 0.3
  vocab_size: 113  # 110 + 3
  cross_coverage: true
  self_coverage: true
  # beam search
  beam_size: 10
  max_len: 150
  alpha: 1.0
  early_stopping: false
  temperature: 1.0
  # training
  learning_rate: 1.0
  patience: 20
  milestones:
    - 300
    - 350
data:
  folder: data/crohme
  test_folder: 2014
  max_size: 320000
  scale_to_limit: true
  train_batch_size: 8
  eval_batch_size: 2
  num_workers: 5
  scale_aug: false

In [ ]:
# Kiểm tra lại config đang dùng
!echo "===== config/crohme.yaml ====="
!cat config/crohme.yaml

## 10. Cấu hình W&B riêng cho notebook này

In [ ]:
# Cấu hình W&B
# Trên Kaggle: Add-ons / Secrets, tạo secret tên WANDB_API_KEY.
# Hai notebook dùng 2 project riêng để không lẫn baseline và CNN-GNN.
import os
os.environ["WANDB_PROJECT"] = "CNN-GNN-HMER-baseline"
os.environ["WANDB_NAME"] = "baseline-crohme-2gpu-t4"
os.environ["WANDB_DIR"] = "/kaggle/working/wandb"
os.environ["WANDB_CACHE_DIR"] = "/kaggle/working/.cache/wandb"
os.environ["WANDB_CONFIG_DIR"] = "/kaggle/working/.config/wandb"

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
    print("Đã lấy WANDB_API_KEY từ Kaggle Secrets.")
except Exception as e:
    print("Chưa lấy được WANDB_API_KEY từ Kaggle Secrets.")
    print("Hãy tạo Kaggle Secret tên WANDB_API_KEY hoặc gán os.environ['WANDB_API_KEY'] thủ công trước cell login.")
    print("Lỗi:", e)

print("WANDB_PROJECT =", os.environ.get("WANDB_PROJECT"))
print("WANDB_NAME =", os.environ.get("WANDB_NAME"))

In [ ]:
# Login W&B
!mkdir -p /kaggle/working/wandb /kaggle/working/.cache/wandb /kaggle/working/.config/wandb
!source /kaggle/working/miniconda/bin/activate tamer && wandb login $WANDB_API_KEY

## 11. Chuẩn bị dữ liệu CROHME

Config đọc dữ liệu ở `data/crohme`, nên cell này giải nén `CROHME.zip` từ repo root vào thư mục model hiện tại.

In [ ]:
# Cách 1: Giải nén CROHME.zip có sẵn trong repo hiện tại
!mkdir -p data
!if [ -f "/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/data/CROHME.zip" ]; then     unzip -o /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/data/CROHME.zip -d data/;   else     echo "Không thấy /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/data/CROHME.zip";     echo "Nếu dùng Kaggle Dataset, chạy cell Cách 2 bên dưới.";   fi

!echo "===== data tree ====="
!find data -maxdepth 3 -type d | sort | head -50

In [ ]:
# Cách 2: Sử dụng từ Kaggle dataset nếu anh upload CROHME.zip riêng
# Sửa path /kaggle/input/crohme-dataset/CROHME.zip nếu dataset của anh có tên khác.
# !mkdir -p data
# !cp /kaggle/input/crohme-dataset/CROHME.zip data/
# !unzip -o data/CROHME.zip -d data/
# !find data -maxdepth 3 -type d | sort | head -50

## 12. Check nhanh repo, data, config, GPU trước khi train

In [ ]:
!echo "===== PWD ====="
!pwd
!echo "===== Repo files ====="
!ls -lh
!echo "===== Config files ====="
!ls -lh config
!echo "===== Data files ====="
!find data -maxdepth 3 | head -80
!echo "===== Eval files ====="
!ls -lh eval || true

In [ ]:
# Kiểm tra CUDA và số GPU T4 Kaggle cấp
!source /kaggle/working/miniconda/bin/activate tamer && python - <<'PY'
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
PY

## 13. Chạy training

In [ ]:
# Chạy training với cấu hình CROHME, mặc định 2 GPU T4
!source /kaggle/working/miniconda/bin/activate tamer && python train.py --config {CONFIG} --trainer.gpus={GPUS}

## 14. Resume training từ checkpoint nếu cần

In [ ]:
# Cell mẫu resume checkpoint. Không chạy nếu chưa có checkpoint.
# Sửa YOUR_CHECKPOINT.ckpt thành checkpoint thật.
# !source /kaggle/working/miniconda/bin/activate tamer && python train.py --config config/crohme.yaml --trainer.resume_from_checkpoint=/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/0-cnn-transformer-baseline/lightning_logs/version_0/checkpoints/YOUR_CHECKPOINT.ckpt --trainer.gpus=2

In [ ]:
# Cell mẫu copy checkpoint từ Kaggle input nếu cần resume.
!mkdir -p /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/0-cnn-transformer-baseline/lightning_logs/version_0/checkpoints
# !cp /kaggle/input/checkpoint/YOUR_CHECKPOINT.ckpt /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/0-cnn-transformer-baseline/lightning_logs/version_0/checkpoints/

## 15. Tìm checkpoint sau training

In [ ]:
# Liệt kê checkpoint sau training
!echo "===== Checkpoints ====="
!find lightning_logs -name "*.ckpt" -type f | sort || true

# Ghi checkpoint cuối cùng tìm được ra best_ckpt.txt để tiện eval/upload
!BEST_CKPT=$(find lightning_logs -name "*.ckpt" -type f | sort | tail -n 1); echo "$BEST_CKPT" | tee best_ckpt.txt
!echo "BEST_CKPT file content:" && cat best_ckpt.txt

## 16. Eval sau training

In [ ]:
# Eval bằng script shell có sẵn trong repo.
# Nếu eval_crohme.sh đã được cấu hình đúng trong project, cell này sẽ chạy trực tiếp.
!source /kaggle/working/miniconda/bin/activate tamer && bash eval/eval_crohme.sh

In [ ]:
# Phương án eval trực tiếp bằng eval/test.py nếu cần truyền checkpoint thủ công.
# Mặc định để comment vì cú pháp tham số phụ thuộc test.py của repo.
# BEST_CKPT=$(cat best_ckpt.txt)
# !source /kaggle/working/miniconda/bin/activate tamer && python eval/test.py --config config/crohme.yaml --checkpoint "$BEST_CKPT"

## 17. Upload checkpoint, log, eval result lên W&B artifact

In [ ]:
# Upload lightning_logs/checkpoints lên W&B Artifact
# Artifact sẽ nằm trong project: CNN-GNN-HMER-baseline
!source /kaggle/working/miniconda/bin/activate tamer && wandb artifact put lightning_logs --name baseline-lightning-logs --type logs || true
!source /kaggle/working/miniconda/bin/activate tamer && wandb artifact put lightning_logs --name baseline-checkpoints --type model || true

# Nếu repo tạo thêm file/thư mục eval result, upload luôn những gì có trong eval.
!source /kaggle/working/miniconda/bin/activate tamer && wandb artifact put eval --name baseline-eval-files --type eval || true

In [ ]:
# Nếu có run W&B offline/local chưa sync, thử sync toàn bộ.
!source /kaggle/working/miniconda/bin/activate tamer && wandb sync --sync-all || true

## 18. Lưu kết quả và môi trường giống template

In [ ]:
# Lưu kết quả training/eval/config để dùng cho phiên sau
!mkdir -p /kaggle/output/baseline_results
!cp -r lightning_logs /kaggle/output/baseline_results/ || true
!cp -r config /kaggle/output/baseline_results/ || true
!cp best_ckpt.txt /kaggle/output/baseline_results/ || true
!cp -r eval /kaggle/output/baseline_results/eval_files_snapshot || true
!find /kaggle/output/baseline_results -maxdepth 3 | head -100

## 19. Nén kết quả ra /kaggle/working

In [ ]:
# Nén thư mục kết quả thành file tar.gz
!tar -czvf baseline_results.tar.gz /kaggle/output/baseline_results

# Chép file nén vào thư mục working
!cp baseline_results.tar.gz /kaggle/working/

# Xác nhận file đã được chép thành công
!ls -lh /kaggle/working/baseline_results.tar.gz